# Zero-Signal Opportunity Model

Two-stage metalabelling pipeline applied to `primary_signal == 0` rows.

**Stage 1** — binary classifier: does a tradeable opportunity exist? (`opportunity_exists ∈ {0,1}`)

**Stage 2** — direction classifier: long or short? (`opportunity_label ∈ {+1,-1}`)

CPCV with purging and embargo prevents lookahead bias throughout.

## 0. Imports and Config

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from itertools import combinations
import json

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    balanced_accuracy_score, accuracy_score, confusion_matrix,
    roc_curve, precision_recall_curve,
)

try:
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE = True
    print("LightGBM available")
except ImportError:
    print("WARNING: lightgbm not installed — pip install lightgbm")
    LGBM_AVAILABLE = False

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ROOT = Path.cwd()
for _c in [ROOT, *ROOT.parents]:
    if (_c / "data" / "labels" / "cl1_merged.csv").exists():
        ROOT = _c
        break

CONFIG = {
    "random_seed":    RANDOM_SEED,
    "n_jobs":         -1,
    "embargo_pct":    0.01,
    "n_cpcv_splits":  4,   # reduced from 6: gives ~90 train rows/fold vs 26
    "n_test_splits":  2,
    "output_path":    ROOT / "data" / "model_outputs",
}
CONFIG["output_path"].mkdir(parents=True, exist_ok=True)

DATA_PATH      = ROOT / "data" / "labels" / "cl1_merged.csv"
OHLCV_PATH     = ROOT / "data" / "raw" / "ohlcv_data.csv"
OUTPUT_PARQUET = CONFIG["output_path"] / "zero_signal_model_output.parquet"
OUTPUT_JSON    = CONFIG["output_path"] / "zero_signal_model_summary.json"

print(f"ROOT:   {ROOT}")
print(f"Data:   {DATA_PATH}")
print(f"Output: {CONFIG['output_path']}")


LightGBM available
ROOT:   C:\Users\madhu\Desktop\Imperial College London\Core Modules\Systematic Trading Startegies\Assignment\systematic-trading-coursework
Data:   C:\Users\madhu\Desktop\Imperial College London\Core Modules\Systematic Trading Startegies\Assignment\systematic-trading-coursework\data\labels\cl1_merged.csv
Output: C:\Users\madhu\Desktop\Imperial College London\Core Modules\Systematic Trading Startegies\Assignment\systematic-trading-coursework\data\model_outputs


## 1. Load Data

In [2]:
df_raw = pd.read_csv(
    DATA_PATH,
    parse_dates=["date", "t0", "t1", "vertical_t1"],
)
print(f"Shape:      {df_raw.shape}")
print(f"Date range: {df_raw['date'].min().date()} → {df_raw['date'].max().date()}")
print(f"\nDtype counts:\n{df_raw.dtypes.value_counts().to_string()}")

_nulls = df_raw.isnull().sum()
print(f"\nNull counts (non-zero only, {(_nulls > 0).sum()} cols):")
print(_nulls[_nulls > 0].to_string())

print("\nprimary_signal value counts:")
print(df_raw["primary_signal"].value_counts().to_string())

display(df_raw[[
    "date", "instrument", "metalabel", "primary_signal",
    "t0", "t1", "entry_price", "position_side", "barrier_outcome"
]].head(5))


Shape:      (618, 208)
Date range: 2020-01-03 → 2022-06-15

Dtype counts:
float64           188
int64              10
object              6
datetime64[ns]      4

Null counts (non-zero only, 53 cols):
hmm_predicted_regime                           2
hmm_volatility_regime                          2
hmm_return_regime                              2
hmm_market_state                               2
hmm_regime_confidence                          2
hmm_prob_low_vol                               2
hmm_prob_normal_vol                            2
hmm_prob_high_vol                              2
hmm_prob_extreme_vol                           2
hmm_prob_downside                              2
hmm_prob_weak                                  2
hmm_prob_positive                              2
hmm_prob_strong_upside                         2
hmm_prob_stress                                2
hmm_prob_upside_breakout                       2
hmm_prob_calm_positive                         2
hmm_prob_calm_n

,date,instrument,metalabel,primary_signal,t0,t1,entry_price,position_side,barrier_outcome
0,2020-01-03,cl1s,0.5,0,2020-01-03,2020-01-08,25.553469,0.0,stop_loss
1,2020-01-06,cl1s,0.5,0,2020-01-06,2020-01-08,25.642633,0.0,stop_loss
2,2020-01-07,cl1s,0.0,-1,2020-01-07,2020-01-08,25.411618,-1.0,take_profit
3,2020-01-08,cl1s,0.5,0,2020-01-08,2020-01-13,24.159275,0.0,stop_loss
4,2020-01-09,cl1s,0.5,0,2020-01-09,2020-01-13,24.139011,0.0,stop_loss


## 2. Filter Zero-Signal Rows

Keep only `primary_signal == 0`. These are days the primary model has no view;
the opportunity model decides whether to trade and in which direction.

In [3]:
total_rows   = len(df_raw)
df_zero      = df_raw[df_raw["primary_signal"] == 0].copy().reset_index(drop=True)
rows_kept    = len(df_zero)
rows_dropped = total_rows - rows_kept

print(f"Total rows:            {total_rows}")
print(f"Rows kept  (sig == 0): {rows_kept}")
print(f"Rows dropped:          {rows_dropped}  ({100 * rows_dropped / total_rows:.1f}%)")
print(f"Date range: {df_zero['date'].min().date()} → {df_zero['date'].max().date()}")

assert rows_kept > 0,                          "No zero-signal rows found"
assert df_zero["primary_signal"].eq(0).all(),  "Non-zero signals remain"


Total rows:            618
Rows kept  (sig == 0): 206
Rows dropped:          412  (66.7%)
Date range: 2020-01-03 → 2022-04-26


## 3. Triple-Barrier Labelling

New labels are generated **in memory only** — no CSVs saved.
A 27-point parameter grid is evaluated; the set with the most balanced
`opportunity_exists` split is selected and applied to the working dataframe.

In [4]:
_ohlcv = pd.read_csv(OHLCV_PATH, parse_dates=["date"])
_cl1s  = (
    _ohlcv[_ohlcv["instrument"].str.lower().eq("cl1s")]
    .sort_values("date")
    .reset_index(drop=True)
)
_CLOSE  = _cl1s.set_index("date")["close"]
_HIGH   = _cl1s.set_index("date")["high"]
_LOW    = _cl1s.set_index("date")["low"]
_DATES  = _HIGH.index

# Compute daily EWMA volatility directly from log-returns of the full price series.
# This gives ~1-3% daily vol for crude oil — the correct scale for barrier widths.
# The feature columns ewma_vol_Xd are annualised (~22%), which would make barriers
# 10x too wide and cause almost all rows to hit the vertical barrier (label = 0).
_log_rets  = np.log(_CLOSE / _CLOSE.shift(1)).dropna()
_DAILY_VOL = _log_rets.ewm(span=20, min_periods=10).std().reindex(_CLOSE.index)

print(f"CL1S OHLCV: {len(_cl1s)} rows  "
      f"{_cl1s['date'].min().date()} to {_cl1s['date'].max().date()}")
print(f"Daily vol:  mean={_DAILY_VOL.mean():.4f}  "
      f"median={_DAILY_VOL.median():.4f}  "
      f"(annualised ~{_DAILY_VOL.mean() * (252**0.5):.1%})")


CL1S OHLCV: 8171 rows  1990-01-02 to 2022-06-30
Daily vol:  mean=0.0214  median=0.0188  (annualised ~34.0%)


In [5]:
def triple_barrier_label(
    df: pd.DataFrame,
    horizon: int,
    pt_mult: float,
    sl_mult: float,
    price_col: str = "entry_price",
) -> pd.DataFrame:
    """Compute triple-barrier labels for each row in df.

    Barriers use daily EWMA vol from _DAILY_VOL (span=20, ~2% for crude oil).
    The feature-matrix vol columns (ewma_vol_Xd) are annualised and would
    produce barriers ~10x too wide, so they are not used here.

    Upper barrier  = price * (1 + pt_mult * daily_vol)
    Lower barrier  = price * (1 - sl_mult * daily_vol)
    Vertical       = horizon trading days from t0

    Returns a copy of df with:
        opportunity_label  (+1 / -1 / 0)
        opportunity_exists (1 if label != 0 else 0)
    """
    labels = []
    for _, row in df.iterrows():
        price = row[price_col]
        vol   = _DAILY_VOL.get(row["t0"])

        if vol is None or pd.isna(vol) or pd.isna(price) or vol <= 0:
            labels.append(0)
            continue

        upper  = price * (1.0 + pt_mult * vol)
        lower  = price * (1.0 - sl_mult * vol)
        future = _DATES[_DATES > row["t0"]][:horizon]

        label = 0
        for dt in future:
            h = _HIGH.get(dt)
            l = _LOW.get(dt)
            if h is None or l is None or pd.isna(h) or pd.isna(l):
                continue
            if h >= upper:
                label = 1
                break
            if l <= lower:
                label = -1
                break

        labels.append(label)

    out = df.copy()
    out["opportunity_label"]  = labels
    out["opportunity_exists"] = (out["opportunity_label"] != 0).astype(int)
    return out


In [6]:
# Parameter grid — kept in memory as a list of dicts, never written to disk.
PARAM_GRID = [
    {"param_id": f"pg_h{h}_pt{pt}_sl{sl}",
     "horizon": h, "pt_mult": pt, "sl_mult": sl}
    for h  in [5, 10, 20]
    for pt in [1.0, 1.5, 2.0]
    for sl in [1.0, 1.5, 2.0]
]
print(f"Pre-computing labels for {len(PARAM_GRID)} parameter sets ...")

# label_cache[param_id] = {"opportunity_exists": np.ndarray,
#                           "opportunity_label":  np.ndarray,
#                           "params": dict}
# All 27 sets are stored so each CPCV fold can select the best one using
# ONLY its training rows — no information from test rows is used.
label_cache = {}
cache_summary = []
for p in PARAM_GRID:
    tmp = triple_barrier_label(
        df_zero, horizon=p["horizon"], pt_mult=p["pt_mult"], sl_mult=p["sl_mult"]
    )
    n_opp   = int(tmp["opportunity_exists"].sum())
    total   = len(tmp)
    balance = round(1.0 - abs(n_opp / total - 0.5) * 2, 4)
    label_cache[p["param_id"]] = {
        "opportunity_exists": tmp["opportunity_label"].map(lambda x: 0 if x == 0 else 1).values,
        "opportunity_label":  tmp["opportunity_label"].values,
        "params":             p,
    }
    cache_summary.append({**p, "n_opportunity": n_opp,
                           "n_no_opp": total - n_opp, "balance_score": balance})

cache_df = pd.DataFrame(cache_summary).sort_values("balance_score", ascending=False)
print(f"Label cache ready: {len(label_cache)} parameter sets")
print("\nTop 10 by balance score (shown for reference — selection happens per fold):")
display(cache_df.head(10))


Pre-computing labels for 27 parameter sets ...


Label cache ready: 27 parameter sets

Top 10 by balance score (shown for reference — selection happens per fold):


,param_id,horizon,pt_mult,sl_mult,n_opportunity,n_no_opp,balance_score
8,pg_h5_pt2.0_sl2.0,5,2.0,2.0,135,71,0.6893
5,pg_h5_pt1.5_sl2.0,5,1.5,2.0,155,51,0.4951
7,pg_h5_pt2.0_sl1.5,5,2.0,1.5,157,49,0.4757
17,pg_h10_pt2.0_sl2.0,10,2.0,2.0,170,36,0.3495
4,pg_h5_pt1.5_sl1.5,5,1.5,1.5,172,34,0.3301
2,pg_h5_pt1.0_sl2.0,5,1.0,2.0,174,32,0.3107
6,pg_h5_pt2.0_sl1.0,5,2.0,1.0,177,29,0.2816
16,pg_h10_pt2.0_sl1.5,10,2.0,1.5,183,23,0.2233
14,pg_h10_pt1.5_sl2.0,10,1.5,2.0,185,21,0.2039
3,pg_h5_pt1.5_sl1.0,5,1.5,1.0,189,17,0.1650


In [7]:
def select_barrier_params_for_fold(train_idx: np.ndarray, label_cache: dict) -> str:
    """Select the barrier parameter set with the most balanced opportunity_exists
    distribution using ONLY the supplied training-fold indices.

    Test indices are never passed to this function — they must not influence
    which parameter set is chosen.

    Returns the param_id of the best parameter set.
    """
    best_id, best_score = None, -1.0
    for param_id, cache in label_cache.items():
        train_labels = cache["opportunity_exists"][train_idx]
        n_opp    = int(train_labels.sum())
        total    = len(train_labels)
        if total == 0:
            continue
        balance  = 1.0 - abs(n_opp / total - 0.5) * 2
        if balance > best_score:
            best_score, best_id = balance, param_id
    return best_id


# Sanity check: function returns a valid key for a dummy training set
_demo = select_barrier_params_for_fold(np.arange(len(df_zero) // 2), label_cache)
assert _demo in label_cache, "select_barrier_params_for_fold returned unknown param_id"
print(f"select_barrier_params_for_fold ready  (demo → {_demo})")
print("Barrier parameter selection will run inside each CPCV fold on train rows only.")


select_barrier_params_for_fold ready  (demo → pg_h5_pt2.0_sl2.0)
Barrier parameter selection will run inside each CPCV fold on train rows only.


## 4. Unsupervised Feature Cleaning

Done once on the full zero-signal set **before** any model split.
No target information is used — purely structural cleaning.

In [8]:
def clean_features(
    X: pd.DataFrame,
    corr_threshold:    float = 0.98,
    missing_threshold: float = 0.50,
) -> tuple:
    """Remove structurally problematic columns from X.

    Removals (in order, every removal logged):
    1. Constant columns          — zero variance, carry no signal
    2. Duplicate columns         — exact byte-identical copies
    3. > 50% missing             — too sparse to impute reliably
    4. Pairwise Spearman |r|>0.98 — near-redundant; keep first encountered

    Returns
    -------
    X_clean    : DataFrame with problem columns removed
    removal_log: dict mapping removed_column_name → reason string
    """
    log  = {}
    cols = list(X.columns)

    # 1. Constant
    for c in cols:
        if X[c].nunique(dropna=True) <= 1:
            log[c] = "constant (nunique <= 1)"
    cols = [c for c in cols if c not in log]

    # 2. Duplicate (hash-based, fast)
    seen = {}
    for c in cols:
        h = pd.util.hash_pandas_object(X[c].fillna(-9999)).sum()
        if h in seen:
            log[c] = f"duplicate of '{seen[h]}'"
        else:
            seen[h] = c
    cols = [c for c in cols if c not in log]

    # 3. High-missing
    for c in cols:
        miss = X[c].isnull().mean()
        if miss > missing_threshold:
            log[c] = f"missing {miss:.1%} > {missing_threshold:.0%}"
    cols = [c for c in cols if c not in log]

    # 4. Pairwise Spearman correlation (via rank transform for speed)
    corr  = X[cols].rank(pct=True).corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    for c in cols:
        if c in log:
            continue
        paired = upper[c][upper[c] >= corr_threshold].index.tolist()
        for drop_c in paired:
            if drop_c not in log:
                log[drop_c] = f"|Spearman|>={corr_threshold:.2f} with '{c}'"

    final = [c for c in cols if c not in log]
    print(f"Started with {len(X.columns):4d} columns")
    print(f"Removed      {len(log):4d} columns")
    print(f"Retained     {len(final):4d} columns")
    print("\nRemoval log:")
    for col, reason in list(log.items())[:60]:
        print(f"  DROP {col!r:50s} ← {reason}")
    if len(log) > 60:
        print(f"  … ({len(log) - 60} more)")
    return X[final].copy(), log


## 5. Feature Matrix

Leakage columns are excluded with an explicit reason comment for each group.
`clean_features` then removes structural redundancy.

In [9]:
# ── Leakage exclusion registry ────────────────────────────────────────────────
# Each set is annotated with its leakage category.

# Forward-looking outcome columns — known only after the position closes
OUTCOME_LEAKAGE = {
    "exit_price",           # price at exit — post-entry
    "raw_return_pct",       # realised P&L — post-entry
    "position_return_pct",  # signed realised P&L — post-entry
    "barrier_outcome",      # which barrier was hit — post-entry
    "triple_barrier_label", # original label — encodes outcome
    "metalabel",            # original metalabel — encodes outcome
}

# New targets — would be direct leakage if used as features
TARGET_COLS = {
    "opportunity_label",    # what we predict in Stage 2
    "opportunity_exists",   # what we predict in Stage 1
}

# Barrier parameter artifacts — structurally linked to target definition
# Including them lets the model reverse-engineer the target boundaries
BARRIER_ARTIFACTS = {
    "t1",            # event end date — encodes when outcome resolved
    "vertical_t1",   # vertical barrier date — encodes chosen horizon
    "position_side", # always 0.0 for zero-signal rows — constant
    "pt",            # take-profit price level — derived from entry + vol
    "sl",            # stop-loss price level   — derived from entry + vol
    "horizon",       # barrier parameter — constant within a param set
    "pt_multiplier", # barrier parameter
    "sl_multiplier", # barrier parameter
    "vol_method",    # barrier parameter
    "vol",           # ewma vol used to SET the existing barriers — param artifact
}

# Identifiers — no predictive content as features
ID_COLS = {
    "date",
    "instrument",
    "primary_signal",  # constant 0 in this subset — no signal
    "t0",              # entry date — same information as date index
}

# String HMM categoricals — already one-hot encoded; avoid double-counting
HMM_STRINGS = {
    "hmm_volatility_regime",
    "hmm_return_regime",
    "hmm_market_state",
}

ALL_EXCLUDED = OUTCOME_LEAKAGE | TARGET_COLS | BARRIER_ARTIFACTS | ID_COLS | HMM_STRINGS

# Use df_zero (no target columns present) so feature columns are
# independent of any particular barrier parameter set.
candidate_cols = [
    c for c in df_zero.columns
    if c not in ALL_EXCLUDED
    and pd.api.types.is_numeric_dtype(df_zero[c])
]

print(f"Total columns in df_zero:      {df_zero.shape[1]}")
print(f"Excluded (all leakage groups): {len(ALL_EXCLUDED)}")
print(f"Candidate numeric features:    {len(candidate_cols)}")

X_raw             = df_zero[candidate_cols].copy()
X_clean, drop_log = clean_features(X_raw)
FEATURE_COLS      = list(X_clean.columns)

print(f"\nFinal feature count: {len(FEATURE_COLS)}")

# y_opportunity and y_direction are fold-specific (selected per fold from
# label_cache using training rows only) — no global y assignments here.

# Leakage guard
assert len(set(FEATURE_COLS) & ALL_EXCLUDED) == 0, "Leakage column in feature set!"


Total columns in df_zero:      208
Excluded (all leakage groups): 25
Candidate numeric features:    185
Started with  185 columns
Removed        68 columns
Retained      117 columns

Removal log:
  DROP 'hmm_prob_chop'                                    ← constant (nunique <= 1)
  DROP 'hmm_market_chop'                                  ← constant (nunique <= 1)
  DROP 'signal_x_hmm_confidence'                          ← constant (nunique <= 1)
  DROP 'signal_x_hmm_prob_stress'                         ← constant (nunique <= 1)
  DROP 'signal_x_hmm_prob_high_or_extreme_vol'            ← constant (nunique <= 1)
  DROP 'signal_x_hmm_prob_positive_or_strong_upside'      ← constant (nunique <= 1)
  DROP 'close'                                            ← duplicate of 'entry_price'
  DROP 'hmm_prob_downside'                                ← duplicate of 'hmm_prob_extreme_vol'
  DROP 'hmm_prob_weak'                                    ← duplicate of 'hmm_prob_low_vol'
  DROP 'hmm_prob_positive

## 6. CPCV Implementation

**Purge**: removes training rows whose label window `[t0, t1]` overlaps any test window.

**Embargo**: removes training rows within `embargo_pct × total_date_span` days
after the end of each test block.

In [10]:
def get_cpcv_splits(
    dates:         pd.Series,
    t0_series:     pd.Series,
    t1_series:     pd.Series,
    n_splits:      int   = 6,
    n_test_splits: int   = 2,
    embargo_pct:   float = 0.01,
) -> list:
    """Build Combinatorial Purged Cross-Validation (CPCV) splits.

    Algorithm
    ---------
    1. Divide N sorted samples into n_splits contiguous time blocks.
    2. Generate all C(n_splits, n_test_splits) test-block combinations.
    3. For each combination:
       a. test_idx  = indices in the selected test blocks.
       b. Purge    : drop train rows where t1 >= earliest test t0
                     (label window bleeds into the test period).
       c. Embargo  : drop train rows within embargo_days of each test-block end.
       d. train_idx = remaining indices.
    4. Skip any fold where train or test would be empty.

    Returns list of (train_idx, test_idx) integer-position arrays.
    """
    n         = len(dates)
    blocks    = np.array_split(np.arange(n), n_splits)
    block_of  = np.empty(n, dtype=int)
    for b_id, b_idx in enumerate(blocks):
        block_of[b_idx] = b_id

    total_days   = max(1, (dates.iloc[-1] - dates.iloc[0]).days)
    embargo_days = max(1, int(np.ceil(embargo_pct * total_days)))

    splits = []
    for test_blocks in combinations(range(n_splits), n_test_splits):
        test_mask    = np.isin(block_of, test_blocks)
        test_idx     = np.where(test_mask)[0]
        if len(test_idx) == 0:
            continue

        # Purge: any train row whose label extends into the test period
        test_t0_min  = t0_series.iloc[test_idx].min()
        purge_mask   = t1_series.values >= test_t0_min

        # Embargo: short buffer after each test block's last date
        embargo_mask = np.zeros(n, dtype=bool)
        for b_id in test_blocks:
            b_end   = dates.iloc[blocks[b_id][-1]]
            emb_end = b_end + pd.Timedelta(days=embargo_days)
            embargo_mask |= (
                (~test_mask)
                & (dates.values > b_end)
                & (dates.values <= emb_end)
            )

        train_idx = np.where(~test_mask & ~purge_mask & ~embargo_mask)[0]
        if len(train_idx) == 0:
            continue

        splits.append((train_idx, test_idx))

    total_possible = len(list(combinations(range(n_splits), n_test_splits)))
    print(f"CPCV: {len(splits)} folds  "
          f"(C({n_splits},{n_test_splits})={total_possible})  "
          f"embargo={embargo_days} days")
    for i, (tr, te) in enumerate(splits[:3]):
        print(f"  fold {i}: train={len(tr)}, test={len(te)}")
    if len(splits) > 3:
        print("  ...")
    return splits


# Use the maximum possible horizon as a conservative t1 for purging.
# This is the worst-case label window — ensures no lookahead regardless
# of which parameter set a fold selects.
MAX_HORIZON    = max(p["horizon"] for p in PARAM_GRID)
conservative_t1 = df_zero["t0"] + pd.Timedelta(days=MAX_HORIZON)

splits = get_cpcv_splits(
    dates         = df_zero["date"],
    t0_series     = df_zero["t0"],
    t1_series     = conservative_t1,
    n_splits      = CONFIG["n_cpcv_splits"],
    n_test_splits = CONFIG["n_test_splits"],
    embargo_pct   = CONFIG["embargo_pct"],
)
assert len(splits) > 0, "No CPCV splits produced"


CPCV: 3 folds  (C(4,2)=6)  embargo=9 days
  fold 0: train=47, test=103
  fold 1: train=47, test=103
  fold 2: train=96, test=102


## 7. Model Definitions

In [11]:
RS = CONFIG["random_seed"]
NJ = CONFIG["n_jobs"]

# Each entry: (name, estimator_instance, param_grid_dict)
_STAGE_MODELS = [
    (
        "logistic_regression",
        LogisticRegression(random_state=RS, max_iter=1000, solver="lbfgs"),
        {"C": [0.01, 0.1, 1.0], "class_weight": ["balanced"]},
    ),
    (
        "random_forest",
        RandomForestClassifier(random_state=RS, n_jobs=NJ),
        {"n_estimators": [100, 300],
         "max_depth":    [3, 5, None],
         "class_weight": ["balanced"]},
    ),
]
if LGBM_AVAILABLE:
    _STAGE_MODELS.append((
        "lgbm",
        LGBMClassifier(random_state=RS, n_jobs=NJ, verbose=-1),
        {"n_estimators":  [100, 300],
         "learning_rate": [0.05, 0.1],
         "max_depth":     [3, 5]},
    ))

STAGE1_MODELS = _STAGE_MODELS   # opportunity filter
STAGE2_MODELS = _STAGE_MODELS   # direction classifier

print("Models:", [m[0] for m in STAGE1_MODELS])


Models: ['logistic_regression', 'random_forest', 'lgbm']


## 8. CPCV Training Loop

In [12]:
def _preprocess(X_tr_raw, X_te_raw):
    """Impute (median) then scale. Both transformers fitted on train only."""
    imp = SimpleImputer(strategy="median")
    scl = StandardScaler()
    X_tr = scl.fit_transform(imp.fit_transform(X_tr_raw))
    X_te = scl.transform(imp.transform(X_te_raw))
    return X_tr, X_te


def _tune_threshold(y_true, y_prob, metric="f1"):
    """Find probability threshold maximising metric on supplied labels.

    Sweeps [0.05, 0.95] in 37 steps; returns best threshold as float.
    Threshold tuned on train-fold data only — never on test.
    """
    best_t, best_s = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 37):
        yp = (y_prob >= t).astype(int)
        if len(np.unique(yp)) < 2:
            continue
        s = (f1_score(y_true, yp, zero_division=0)
             if metric == "f1"
             else balanced_accuracy_score(y_true, yp))
        if s > best_s:
            best_s, best_t = s, float(t)
    return best_t


def _fit_stage(X_tr, y_tr, X_te, estimator, param_grid, cv_scoring):
    """GridSearchCV on train, refit best estimator, predict test.

    Returns (y_prob, y_pred_05, best_estimator, best_params_dict).
    Preprocessing is assumed already done by the caller.
    """
    assert not np.isnan(X_tr).any(), "NaNs in X_train"
    assert not np.isnan(X_te).any(), "NaNs in X_test"
    assert len(np.unique(y_tr)) >= 2, "Single class in train fold"

    gs = GridSearchCV(
        estimator  = estimator,
        param_grid = param_grid,
        cv         = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED),
        scoring    = cv_scoring,
        n_jobs     = 1,
        refit      = True,
    )
    gs.fit(X_tr, y_tr)
    best = gs.best_estimator_
    prob = best.predict_proba(X_te)[:, 1]
    return prob, (prob >= 0.5).astype(int), best, gs.best_params_


In [13]:
X_all        = df_zero[FEATURE_COLS].values
fold_results = []

for fold_id, (train_idx, test_idx) in enumerate(splits):
    print(f"\n── Fold {fold_id}  train={len(train_idx)}  test={len(test_idx)} ──")

    best_pid     = select_barrier_params_for_fold(train_idx, label_cache)
    fold_cache   = label_cache[best_pid]
    y_opp_all    = fold_cache["opportunity_exists"]
    y_dir_all    = fold_cache["opportunity_label"]
    fold_horizon = int(fold_cache["params"]["horizon"])
    print(f"  barrier params: {best_pid}  "
          f"opp_rate={y_opp_all[train_idx].mean():.2f}")

    X_tr, X_te = _preprocess(X_all[train_idx], X_all[test_idx])
    y_opp_tr   = y_opp_all[train_idx]
    y_opp_te   = y_opp_all[test_idx]

    for model_name, estimator, param_grid in STAGE1_MODELS:

        # ── Stage 1: opportunity exists? ─────────────────────────────────────
        if len(np.unique(y_opp_tr)) < 2:
            print(f"  SKIP {model_name} s1 — single class in train")
            continue

        prob1, _, est1, hp1 = _fit_stage(
            X_tr, y_opp_tr, X_te, estimator, param_grid, "roc_auc"
        )
        opp_thr = _tune_threshold(y_opp_tr, est1.predict_proba(X_tr)[:, 1])
        pred1   = (prob1 >= opp_thr).astype(int)

        # ── Stage 2: direction (+1 / -1) ─────────────────────────────────────
        dir_tr_mask  = y_dir_all[train_idx] != 0
        dir_te_mask  = y_dir_all[test_idx]  != 0
        y_dir_tr_bin = (y_dir_all[train_idx][dir_tr_mask] > 0).astype(int)

        prob2_full = np.full(len(test_idx), 0.5)
        pred2_full = np.zeros(len(test_idx), dtype=int)
        dir_thr    = 0.5
        hp2        = {}
        est2       = None

        if dir_tr_mask.sum() >= 4 and len(np.unique(y_dir_tr_bin)) >= 2:
            X_dir_tr, X_dir_te = _preprocess(
                X_all[train_idx][dir_tr_mask],
                X_all[test_idx][dir_te_mask] if dir_te_mask.sum() > 0
                else X_all[train_idx][dir_tr_mask][:1],
            )
            assert not np.isnan(X_dir_tr).any(), "NaNs in direction X_train"

            prob2, _, est2, hp2 = _fit_stage(
                X_dir_tr, y_dir_tr_bin,
                X_dir_te, estimator, param_grid, "f1_macro"
            )
            dir_thr = _tune_threshold(y_dir_tr_bin,
                                      est2.predict_proba(X_dir_tr)[:, 1])

            if dir_te_mask.sum() > 0:
                te_positions = np.where(dir_te_mask)[0]
                prob2_full[te_positions] = prob2
                pred2_full[te_positions] = (prob2 >= dir_thr).astype(int)
        else:
            print(f"  SKIP {model_name} s2 — {dir_tr_mask.sum()} direction train rows")

        imp_s1 = (dict(zip(FEATURE_COLS, est1.feature_importances_))
                  if hasattr(est1, "feature_importances_") else None)
        imp_s2 = (dict(zip(FEATURE_COLS, est2.feature_importances_))
                  if est2 is not None and hasattr(est2, "feature_importances_") else None)

        fold_results.append({
            "fold_id":        fold_id,
            "model_name":     model_name,
            "train_idx":      train_idx,
            "test_idx":       test_idx,
            "y_opp_true":     y_opp_te,
            "y_opp_prob":     prob1,
            "y_opp_pred":     pred1,
            "opp_threshold":  opp_thr,
            "y_dir_true":     y_dir_all[test_idx],
            "y_dir_prob":     prob2_full,
            "y_dir_pred_bin": pred2_full,
            "dir_threshold":  dir_thr,
            "hp_stage1":      hp1,
            "hp_stage2":      hp2,
            "importances_s1": imp_s1,
            "importances_s2": imp_s2,
            "param_set_id":   best_pid,
            "fold_horizon":   fold_horizon,   # stored for forward-return computation
            "fold_y_opp":     y_opp_all,
            "fold_y_dir":     y_dir_all,
        })
        print(f"  {model_name:22s}  opp_thr={opp_thr:.2f}  dir_thr={dir_thr:.2f}  "
              f"dir_train_rows={dir_tr_mask.sum()}")

print(f"\nTotal fold-model records: {len(fold_results)}")
assert len(fold_results) > 0, "No fold results produced"



── Fold 0  train=47  test=103 ──
  barrier params: pg_h5_pt2.0_sl2.0  opp_rate=0.64


  logistic_regression     opp_thr=0.32  dir_thr=0.47  dir_train_rows=30


  random_forest           opp_thr=0.45  dir_thr=0.20  dir_train_rows=30


  lgbm                    opp_thr=0.45  dir_thr=0.50  dir_train_rows=30

── Fold 1  train=47  test=103 ──
  barrier params: pg_h5_pt2.0_sl2.0  opp_rate=0.64


  logistic_regression     opp_thr=0.32  dir_thr=0.47  dir_train_rows=30


  random_forest           opp_thr=0.45  dir_thr=0.20  dir_train_rows=30


  lgbm                    opp_thr=0.45  dir_thr=0.50  dir_train_rows=30

── Fold 2  train=96  test=102 ──
  barrier params: pg_h5_pt2.0_sl2.0  opp_rate=0.59


  logistic_regression     opp_thr=0.53  dir_thr=0.53  dir_train_rows=57


  random_forest           opp_thr=0.50  dir_thr=0.27  dir_train_rows=57


  lgbm                    opp_thr=0.27  dir_thr=0.37  dir_train_rows=57

Total fold-model records: 9


## 9. Decision Logic

For each test row:
- If `pred_opportunity_prob < opportunity_threshold` → signal = **0** (Stage 1 gate)
- Else if `pred_direction_prob >= direction_threshold` → signal = **+1 or -1** (confident direction)
- Else → signal = **0** (Stage 2 gate: direction not confident)

In [14]:
def apply_decision_logic(res: dict) -> np.ndarray:
    """Combine Stage 1 and Stage 2 outputs into a final integer signal.

    y_dir_pred_bin = 1 -> +1 (long)
    y_dir_pred_bin = 0 -> -1 (short)
    Both thresholds tuned on train data only.
    """
    opp_prob = res["y_opp_prob"]
    dir_prob = res["y_dir_prob"]
    dir_pred = res["y_dir_pred_bin"]
    opp_thr  = res["opp_threshold"]
    dir_thr  = res["dir_threshold"]

    final = np.zeros(len(opp_prob), dtype=int)
    for i in range(len(opp_prob)):
        if opp_prob[i] < opp_thr:
            final[i] = 0
        elif dir_prob[i] >= dir_thr:
            final[i] = 1 if dir_pred[i] == 1 else -1
    return final


for res in fold_results:
    res["final_zero_signal"] = apply_decision_logic(res)


def _forward_signed_return_pct(t0, entry_price, horizon, signal):
    """Compute the signed % return for a hypothetical trade.

    position_return_pct in the source data is 0 for all zero-signal rows
    because the primary strategy held no position on those days.
    This function simulates: enter at entry_price on t0, hold horizon
    trading days, exit at closing price.

    Returns signal * (exit - entry) / entry * 100, so positive means
    the trade was profitable regardless of direction.
    """
    if signal == 0 or pd.isna(entry_price) or entry_price <= 0:
        return 0.0
    future = _DATES[_DATES > t0][:horizon]
    if len(future) < horizon:
        return 0.0
    exit_px = _CLOSE.get(future[-1])
    if exit_px is None or pd.isna(exit_px):
        return 0.0
    return float(signal * 100.0 * (exit_px - entry_price) / entry_price)


# Build OOF prediction dataframe.
oof_rows = []
for res in fold_results:
    fold_y_opp   = res["fold_y_opp"]
    fold_y_dir   = res["fold_y_dir"]
    fold_horizon = res["fold_horizon"]

    for local_i, global_i in enumerate(res["test_idx"]):
        row    = df_zero.iloc[global_i]
        signal = int(res["final_zero_signal"][local_i])

        # Compute forward return using actual OHLCV prices.
        # This is the correct evaluation return for zero-signal rows.
        eval_ret = _forward_signed_return_pct(
            row["t0"], row["entry_price"], fold_horizon, signal
        )

        oof_rows.append({
            "date":                  row["date"],
            "instrument":            row["instrument"],
            "primary_signal":        int(row["primary_signal"]),
            "parameter_set_id":      res["param_set_id"],
            "opportunity_label":     int(fold_y_dir[global_i]),
            "opportunity_exists":    int(fold_y_opp[global_i]),
            "pred_opportunity_prob": float(res["y_opp_prob"][local_i]),
            "pred_direction":        int(res["final_zero_signal"][local_i]),
            "pred_direction_prob":   float(res["y_dir_prob"][local_i]),
            "final_zero_signal":     signal,
            "fold_id":               int(res["fold_id"]),
            "model_type":            res["model_name"],
            "_eval_return_pct":      eval_ret,
        })

oof_df = (pd.DataFrame(oof_rows)
          .sort_values(["date", "model_type"])
          .reset_index(drop=True))

print(f"OOF rows: {len(oof_df)}")
print("\nopportunity_exists distribution (across all folds):")
print(oof_df.groupby("model_type")["opportunity_exists"].value_counts()
      .unstack(fill_value=0).to_string())
print("\nfinal_zero_signal distribution by model:")
print(oof_df.groupby("model_type")["final_zero_signal"]
      .value_counts().unstack(fill_value=0).to_string())
print("\nNon-zero signals with non-zero returns:")
active = oof_df[oof_df["final_zero_signal"] != 0]
print(f"  trades: {len(active)}  "
      f"mean_return: {active['_eval_return_pct'].mean():.3f}%  "
      f"hit_rate: {(active['_eval_return_pct'] > 0).mean():.2%}")


OOF rows: 924

opportunity_exists distribution (across all folds):
opportunity_exists    0    1
model_type                  
lgbm                 98  210
logistic_regression  98  210
random_forest        98  210

final_zero_signal distribution by model:
final_zero_signal    -1    0    1
model_type                       
lgbm                 78  199   31
logistic_regression  63  125  120
random_forest        87   53  168

Non-zero signals with non-zero returns:
  trades: 547  mean_return: 0.682%  hit_rate: 55.39%


## 10. Evaluation

### 10a. ML Metrics

In [15]:
ml_rows = []
for res in fold_results:
    y_t = res["y_opp_true"]; y_p = res["y_opp_prob"]; y_d = res["y_opp_pred"]
    auc = roc_auc_score(y_t, y_p) if len(np.unique(y_t)) >= 2 else np.nan
    ml_rows.append({
        "fold_id": res["fold_id"], "model": res["model_name"], "stage": "opportunity",
        "roc_auc": auc,
        "f1":        f1_score(y_t, y_d, zero_division=0),
        "precision": precision_score(y_t, y_d, zero_division=0),
        "recall":    recall_score(y_t, y_d, zero_division=0),
        "bal_acc":   balanced_accuracy_score(y_t, y_d),
        "n_test":    len(y_t),
    })
    # Direction metrics (on non-zero true labels only)
    dir_mask = res["y_dir_true"] != 0
    if dir_mask.sum() >= 4:
        yt2  = (res["y_dir_true"][dir_mask] > 0).astype(int)
        yp2  = res["y_dir_pred_bin"][dir_mask]
        ypr2 = res["y_dir_prob"][dir_mask]
        auc2 = roc_auc_score(yt2, ypr2) if len(np.unique(yt2)) >= 2 else np.nan
        ml_rows.append({
            "fold_id": res["fold_id"], "model": res["model_name"], "stage": "direction",
            "roc_auc": auc2,
            "f1":        f1_score(yt2, yp2, average="macro", zero_division=0),
            "precision": precision_score(yt2, yp2, average="macro", zero_division=0),
            "recall":    recall_score(yt2, yp2, average="macro", zero_division=0),
            "bal_acc":   balanced_accuracy_score(yt2, yp2),
            "n_test":    int(dir_mask.sum()),
        })

ml_df = pd.DataFrame(ml_rows)
summary_ml = (
    ml_df.groupby(["model", "stage"])[["roc_auc", "f1", "precision", "recall", "bal_acc"]]
    .agg(["mean", "std"])
    .round(4)
)
print("ML Metrics — mean ± std across CPCV folds")
display(summary_ml)


ML Metrics — mean ± std across CPCV folds


roc_auc              f1         precision  \
                                   mean     std    mean     std      mean   
model               stage                                                   
lgbm                direction    0.5084  0.0146  0.3887  0.1363    0.3325   
                    opportunity  0.5329  0.0527  0.7478  0.0034    0.6886   
logistic_regression direction    0.4547  0.0613  0.4436  0.0593    0.4572   
                    opportunity  0.4364  0.1525  0.6796  0.1329    0.6368   
random_forest       direction    0.4148  0.0638  0.3705  0.0487    0.3799   
                    opportunity  0.5775  0.0768  0.7729  0.0576    0.6849   

                                         recall         bal_acc          
                                    std    mean     std    mean     std  
model               stage                                                
lgbm                direction    0.1849  0.5146  0.0253  0.5146  0.0253  
                    opportunity  0.0326  0.8213  0.0466  0.5106  0.0414  
logistic_regression direction    0.0511  0.4755  0.0349  0.4755  0.0349  
                    opportunity  0.0443  0.7531  0.2499  0.4336  0.1133  
random_forest       direction    0.1117  0.4503  0.0427  0.4503  0.0427  
                    opportunity  0.0428  0.9001  0.1475  0.5087  0.0476

In [16]:
# ROC and PR curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = {"logistic_regression": "steelblue", "random_forest": "darkorange", "lgbm": "green"}
for res in fold_results:
    yt = res["y_opp_true"]; yp = res["y_opp_prob"]
    if len(np.unique(yt)) < 2:
        continue
    c = colors.get(res["model_name"], "grey")
    fpr, tpr, _ = roc_curve(yt, yp)
    pr, rc, _   = precision_recall_curve(yt, yp)
    axes[0].plot(fpr, tpr, alpha=0.35, lw=1, color=c, label=res["model_name"])
    axes[1].plot(rc, pr,  alpha=0.35, lw=1, color=c)

axes[0].plot([0,1],[0,1],"k--",lw=1)
axes[0].set(xlabel="FPR", ylabel="TPR",
            title="ROC — Opportunity Filter (all folds)")
axes[1].set(xlabel="Recall", ylabel="Precision",
            title="PR Curve — Opportunity Filter (all folds)")
handles = [plt.Line2D([0],[0],color=c,label=n) for n,c in colors.items()
           if n in oof_df["model_type"].values]
axes[0].legend(handles=handles, fontsize=8)
plt.tight_layout()
plt.savefig(CONFIG["output_path"] / "roc_pr_curves.png", dpi=120, bbox_inches="tight")
plt.show()

# Confusion matrices
models_present = oof_df["model_type"].unique()
fig, axes = plt.subplots(1, len(models_present),
                         figsize=(5 * len(models_present), 4))
if len(models_present) == 1:
    axes = [axes]
for ax, mn in zip(axes, models_present):
    sub = oof_df[oof_df["model_type"] == mn]
    cm  = confusion_matrix(sub["opportunity_exists"],
                           (sub["pred_opportunity_prob"] >= 0.5).astype(int))
    sns.heatmap(cm, annot=True, fmt="d", ax=ax, cmap="Blues")
    ax.set(title=f"{mn}\nOpportunity Confusion Matrix",
           xlabel="Predicted", ylabel="True")
plt.tight_layout()
plt.savefig(CONFIG["output_path"] / "confusion_matrices.png", dpi=120, bbox_inches="tight")
plt.show()


### 10b. Trading Metrics

`_eval_return_pct` (= `position_return_pct`) is used here for evaluation only — it was excluded from all feature sets.

In [17]:
def compute_trading_metrics(
    signals:    np.ndarray,
    returns_pct: np.ndarray,
    tc_per_side: float = 0.001,
    ann_factor:  int   = 252,
) -> dict:
    """Compute trading performance metrics from OOF signals.

    Parameters
    ----------
    signals      : array of -1 / 0 / +1 per row
    returns_pct  : position_return_pct for evaluation only (never a feature)
    tc_per_side  : one-way transaction cost as fraction of notional (default 0.1%)
    ann_factor   : trading days per year

    Returns labelled metrics dict.
    """
    active  = signals != 0
    n_trades = int(active.sum())
    if n_trades == 0:
        return {"n_trades": 0, "ann_return": np.nan, "ann_vol": np.nan,
                "sharpe": np.nan, "sharpe_tc": np.nan,
                "max_drawdown": np.nan, "hit_rate": np.nan, "turnover": np.nan}

    rets = returns_pct[active] / 100.0  # % → fraction

    ann_ret = float(np.mean(rets) * ann_factor)
    ann_vol = float(np.std(rets, ddof=1) * np.sqrt(ann_factor))
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else np.nan

    tc_rets    = rets - 2 * tc_per_side
    ann_ret_tc = float(np.mean(tc_rets) * ann_factor)
    sharpe_tc  = ann_ret_tc / ann_vol if ann_vol > 0 else np.nan

    cum   = np.cumprod(1 + rets)
    peak  = np.maximum.accumulate(cum)
    max_dd = float(((cum - peak) / peak).min())

    return {
        "n_trades":    n_trades,
        "ann_return":  round(ann_ret,   4),
        "ann_vol":     round(ann_vol,   4),
        "sharpe":      round(sharpe,    4),
        "sharpe_tc":   round(sharpe_tc, 4),
        "max_drawdown": round(max_dd,   4),
        "hit_rate":    round(float((rets > 0).mean()), 4),
        "turnover":    round(float(n_trades / len(signals)), 4),
    }


trade_rows = []
for mn in oof_df["model_type"].unique():
    sub = oof_df[oof_df["model_type"] == mn].sort_values("date")
    m   = compute_trading_metrics(
        sub["final_zero_signal"].values,
        sub["_eval_return_pct"].values,
    )
    m["model"] = mn
    trade_rows.append(m)

trade_df = pd.DataFrame(trade_rows).set_index("model")
print("Trading Metrics (OOF, per model):")
display(trade_df.T)


Trading Metrics (OOF, per model):


model,lgbm,logistic_regression,random_forest
n_trades,109.0000,183.0000,255.0000
ann_return,1.1806,1.6282,2.0110
ann_vol,0.7486,0.7554,0.8345
sharpe,1.5770,2.1553,2.4099
sharpe_tc,0.9038,1.4882,1.8059
max_drawdown,-0.2597,-0.5535,-0.6466
hit_rate,0.4954,0.5574,0.5765
turnover,0.3539,0.5942,0.8279


## 11. Feature Importance

Averaged across CPCV folds for tree-based models.

In [18]:
def plot_feature_importance(fold_results: list, stage: str = "s1", top_n: int = 20):
    """Average feature importances across folds and plot horizontal bar chart.

    Parameters
    ----------
    stage  : 's1' (opportunity) or 's2' (direction)
    top_n  : number of top features to display
    """
    key   = "importances_s1" if stage == "s1" else "importances_s2"
    label = "Stage 1 – Opportunity" if stage == "s1" else "Stage 2 – Direction"

    by_model = {}
    for res in fold_results:
        if res[key] is not None:
            by_model.setdefault(res["model_name"], []).append(res[key])

    if not by_model:
        print(f"No importances for {label}")
        return

    fig, axes = plt.subplots(1, len(by_model),
                             figsize=(9 * len(by_model), 6))
    if len(by_model) == 1:
        axes = [axes]

    for ax, (mn, imp_list) in zip(axes, by_model.items()):
        avg = pd.DataFrame(imp_list).mean().sort_values(ascending=False).head(top_n)
        avg.sort_values().plot.barh(ax=ax, color="steelblue")
        ax.set(title=f"{mn}\n{label} — top {top_n}", xlabel="Mean importance")
        ax.axvline(0, color="black", lw=0.8)

    plt.tight_layout()
    out = CONFIG["output_path"] / f"feature_importance_{stage}.png"
    plt.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}")


plot_feature_importance(fold_results, "s1")
plot_feature_importance(fold_results, "s2")


Saved: C:\Users\madhu\Desktop\Imperial College London\Core Modules\Systematic Trading Startegies\Assignment\systematic-trading-coursework\data\model_outputs\feature_importance_s1.png


Saved: C:\Users\madhu\Desktop\Imperial College London\Core Modules\Systematic Trading Startegies\Assignment\systematic-trading-coursework\data\model_outputs\feature_importance_s2.png


## 12. Save Outputs

Exactly **two files**. No intermediate CSVs. No per-fold files.

In [19]:
# ── File 1: predictions parquet ───────────────────────────────────────────────
SAVE_COLS = [
    "date", "instrument", "primary_signal", "parameter_set_id",
    "opportunity_label", "opportunity_exists",
    "pred_opportunity_prob", "pred_direction", "pred_direction_prob",
    "final_zero_signal", "fold_id", "model_type",
]
# _eval_return_pct is deliberately excluded — it is evaluation-only,
# not a model output that should be distributed.

oof_save = oof_df[SAVE_COLS].copy()
oof_save.to_parquet(OUTPUT_PARQUET, index=False)
print(f"Saved parquet: {OUTPUT_PARQUET}  ({len(oof_save)} rows)")

# ── File 2: summary JSON ──────────────────────────────────────────────────────
best_model = (
    ml_df[ml_df["stage"] == "opportunity"]
    .groupby("model")["roc_auc"].mean()
    .idxmax()
)
best_res   = [r for r in fold_results if r["model_name"] == best_model]
best_hp_s1 = best_res[0]["hp_stage1"] if best_res else {}
best_hp_s2 = best_res[0]["hp_stage2"] if best_res else {}

# Most frequently selected barrier param set across the best model's folds
from collections import Counter
param_counts     = Counter(r["param_set_id"] for r in best_res)
most_common_pid  = param_counts.most_common(1)[0][0]
most_common_params = label_cache[most_common_pid]["params"]

summary = {
    "best_model_type":         best_model,
    "best_hyperparameters_s1": {str(k): str(v) for k, v in best_hp_s1.items()},
    "best_hyperparameters_s2": {str(k): str(v) for k, v in best_hp_s2.items()},
    "best_barrier_parameters": {
        "param_id":           most_common_pid,
        "horizon":            int(most_common_params["horizon"]),
        "pt_mult":            float(most_common_params["pt_mult"]),
        "sl_mult":            float(most_common_params["sl_mult"]),
        "selection_note":     "most frequent across CPCV folds for best model",
        "fold_param_counts":  dict(param_counts),
    },
    "opportunity_threshold": float(np.mean(
        [r["opp_threshold"] for r in best_res])),
    "direction_threshold":   float(np.mean(
        [r["dir_threshold"] for r in best_res])),
    "cpcv_ml_metrics": (
        ml_df.groupby(["model", "stage"])
        [["roc_auc", "f1", "precision", "recall", "bal_acc"]]
        .mean().round(4).reset_index().to_dict(orient="records")
    ),
    "trading_metrics": trade_df.reset_index().to_dict(orient="records"),
    "n_features_used":    len(FEATURE_COLS),
    "n_zero_signal_rows": int(rows_kept),
    "n_cpcv_folds":       len(splits),
}

with open(OUTPUT_JSON, "w") as fh:
    json.dump(summary, fh, indent=2, default=str)
print(f"Saved JSON:    {OUTPUT_JSON}")

# ── Validation ────────────────────────────────────────────────────────────────
assert OUTPUT_PARQUET.exists(),                    "Parquet missing"
assert OUTPUT_JSON.exists(),                       "JSON missing"
assert "_eval_return_pct" not in oof_save.columns, "Eval column leaked into save"
assert set(SAVE_COLS) <= set(oof_save.columns),    "Missing save columns"
assert len(set(FEATURE_COLS) & ALL_EXCLUDED) == 0, "Leakage in feature set"

saved_files = list(CONFIG["output_path"].glob("zero_signal_model_*"))
assert len(saved_files) == 2, f"Expected 2 output files, found {len(saved_files)}"

print("\nValidation passed.")
print(f"  {OUTPUT_PARQUET.name}")
print(f"  {OUTPUT_JSON.name}")


Saved parquet: C:\Users\madhu\Desktop\Imperial College London\Core Modules\Systematic Trading Startegies\Assignment\systematic-trading-coursework\data\model_outputs\zero_signal_model_output.parquet  (924 rows)
Saved JSON:    C:\Users\madhu\Desktop\Imperial College London\Core Modules\Systematic Trading Startegies\Assignment\systematic-trading-coursework\data\model_outputs\zero_signal_model_summary.json

Validation passed.
  zero_signal_model_output.parquet
  zero_signal_model_summary.json
